# 03 — Hypothesis Tests (Phase 4)

Formal testing of the relationships previewed in `02_descriptive_stats.ipynb`.
Every test reports: statistic, p-value, effect size with interpretation, and 95% CI where computable.

**Locked decisions** (from Phase 1 & 2 state):
- Delivered-only filter for delivery and satisfaction tests
- `review_score` treated as ordinal — no Pearson, no t-test on means
- State not controlled separately (delivery-mediated, ρ = −0.796)
- Price = `items_price_total`, freight excluded

**Execution order:** Q1 → §4-A → §4-B → Q2 → Q3 → Q4a/b → §1 ANOVA.

---

## Q1 — Does order price correlate with delivery time?

Spearman rank correlation on `items_price_total` vs `delivery_days`, delivered orders only.
H₀: ρ = 0. Two-sided. 95% CI via 2000-resample bootstrap.

In [40]:
# =========================================================
# Q1: Does order price correlate with delivery time?
# Spearman rank correlation, delivered-only, freight excluded
# =========================================================
from scipy import stats

# Delivered-only filter (decision #1)
delivered = orders[orders["order_status"] == "delivered"].copy()

# Defensive: drop rows missing either variable
q1 = delivered[["items_price_total", "delivery_days"]].dropna()
print(f"Q1 sample size: {len(q1):,} (dropped {len(delivered) - len(q1)} rows with NaN)")

# Spearman rank correlation
rho, pval = stats.spearmanr(q1["items_price_total"], q1["delivery_days"])
print(f"\nSpearman ρ = {rho:+.4f}")
print(f"p-value    = {pval:.2e}")

# Bootstrap 95% CI on ρ (2000 resamples, paired)
def spearman_stat(x, y):
    return stats.spearmanr(x, y).statistic

rng = np.random.default_rng(42)
boot = stats.bootstrap(
    (q1["items_price_total"].values, q1["delivery_days"].values),
    statistic=spearman_stat,
    n_resamples=2000,
    paired=True,
    confidence_level=0.95,
    random_state=rng,
    method="percentile",
)
ci_lo, ci_hi = boot.confidence_interval
print(f"95% CI     = [{ci_lo:+.4f}, {ci_hi:+.4f}]  (bootstrap, 2000 resamples)")

# Interpretation using Cohen-style thresholds for ρ
abs_rho = abs(rho)
if   abs_rho < 0.10: label = "trivial"
elif abs_rho < 0.30: label = "weak"
elif abs_rho < 0.50: label = "moderate"
else:                label = "strong"
print(f"\nEffect size: {label} ({rho:+.3f})")

Q1 sample size: 96,470 (dropped 8 rows with NaN)

Spearman ρ = +0.1017
p-value    = 4.49e-220
95% CI     = [+0.0953, +0.1078]  (bootstrap, 2000 resamples)

Effect size: weak (+0.102)


In [41]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Decile bin for the line panel
q1_binned = q1.copy()
q1_binned["price_decile"] = pd.qcut(q1_binned["items_price_total"], 10, labels=False) + 1
decile_stats = q1_binned.groupby("price_decile").agg(
    price_median=("items_price_total", "median"),
    delivery_median=("delivery_days", "median"),
    delivery_q25=("delivery_days", lambda s: s.quantile(0.25)),
    delivery_q75=("delivery_days", lambda s: s.quantile(0.75)),
).reset_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Scatter (log x) — ρ = {rho:+.3f}",
        "Median delivery days by price decile (IQR shaded)"
    ),
    horizontal_spacing=0.12,
)

# Left: scatter, sampled to 5k for render speed
sample = q1.sample(5000, random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample["items_price_total"], y=sample["delivery_days"],
        mode="markers", marker=dict(size=3, opacity=0.25),
        showlegend=False,
    ),
    row=1, col=1,
)

# Right: decile medians with IQR band
fig.add_trace(
    go.Scatter(
        x=decile_stats["price_median"], y=decile_stats["delivery_q75"],
        mode="lines", line=dict(width=0), showlegend=False,
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(
        x=decile_stats["price_median"], y=decile_stats["delivery_q25"],
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(100,100,200,0.2)",
        name="IQR", showlegend=True,
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(
        x=decile_stats["price_median"], y=decile_stats["delivery_median"],
        mode="lines+markers", line=dict(width=3), name="Median",
    ),
    row=1, col=2,
)

fig.update_xaxes(type="log", title_text="Order item-price total (BRL, log)", row=1, col=1)
fig.update_xaxes(type="log", title_text="Decile median price (BRL, log)", row=1, col=2)
fig.update_yaxes(title_text="Delivery days", row=1, col=1)
fig.update_yaxes(title_text="Delivery days", row=1, col=2)
fig.update_layout(
    title_text="Q1 — Price vs delivery time (delivered orders, n≈96k)",
    height=500, width=1100, template="plotly_white",
)

fig.show()
fig.write_image(PROJECT_ROOT / "reports" / "figures" / "07_q1_price_vs_delivery.png", scale=2)

### Q1 — Result

**Test:** Spearman ρ on `items_price_total` vs `delivery_days`, delivered orders only.
**Sample:** n = 96,470 (8 rows dropped for missing values).

| Metric | Value |
|---|---|
| ρ | +0.102 |
| 95% CI | [+0.095, +0.108] |
| p-value | 4.5 × 10⁻²²⁰ |
| ρ² | 0.010 |
| Effect size | weak, effectively trivial |

**Interpretation.** Price and delivery time are functionally uncorrelated. The CI
lower bound (+0.095) sits below the conventional 0.10 "weak" threshold — this is
honestly better read as trivial than weak. Price explains ~1% of delivery-time
variance. The vanishingly small p-value is a textbook demonstration that significance
carries no information about magnitude at n ≈ 100k: at this sample size any ρ ≠ 0
will register p ≈ 0.

The decile-median panel shows the signal most cleanly: median delivery climbs
from ~9.3 days (cheapest decile) to ~11.3 days (most expensive decile), a ~2-day
spread across a >20× price range. Likely compositional — expensive orders skew
toward heavier/bulkier items on different carrier tiers — not a systematic
slow-lane for premium orders.

**Methodological note.** Phase 2 preview suggested ρ ≈ +0.14; the final value is
lower because we correlated against `items_price_total` (freight excluded) rather
than payment total. Including freight would have been partially auto-correlated
with delivery time — freight is itself a function of the delivery process.

**Carry-forward.** Price is not a meaningful delivery predictor. No need to
condition on price in §4-A (delivery → review) or §4-B (on_time → review).

In [42]:
# =========================================================
# §4-A: Does delivery time correlate with review score?
# Spearman rank correlation, delivered + reviewed only
# =========================================================

# Delivered + reviewed only
a = delivered[["delivery_days", "review_score"]].dropna()
print(f"§4-A sample size: {len(a):,} (dropped {len(delivered) - len(a)} rows)")

# Spearman
rho_a, pval_a = stats.spearmanr(a["delivery_days"], a["review_score"])
print(f"\nSpearman ρ = {rho_a:+.4f}")
print(f"p-value    = {pval_a:.2e}")

# Bootstrap 95% CI
rng = np.random.default_rng(42)
boot_a = stats.bootstrap(
    (a["delivery_days"].values, a["review_score"].values),
    statistic=spearman_stat,
    n_resamples=2000,
    paired=True,
    confidence_level=0.95,
    random_state=rng,
    method="percentile",
)
ci_lo_a, ci_hi_a = boot_a.confidence_interval
print(f"95% CI     = [{ci_lo_a:+.4f}, {ci_hi_a:+.4f}]  (bootstrap, 2000 resamples)")

# Effect-size label
abs_rho_a = abs(rho_a)
if   abs_rho_a < 0.10: label_a = "trivial"
elif abs_rho_a < 0.30: label_a = "weak"
elif abs_rho_a < 0.50: label_a = "moderate"
else:                  label_a = "strong"
print(f"\nEffect size: {label_a} ({rho_a:+.3f})")
print(f"ρ² = {rho_a**2:.3f} ({rho_a**2 * 100:.1f}% of rank-variance shared)")

§4-A sample size: 95,824 (dropped 654 rows)

Spearman ρ = -0.2346
p-value    = 0.00e+00
95% CI     = [-0.2407, -0.2284]  (bootstrap, 2000 resamples)

Effect size: weak (-0.235)
ρ² = 0.055 (5.5% of rank-variance shared)


In [43]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Left: box plot of delivery_days by review_score (ordinal framing)
# Right: binned delivery days → mean review score
a_binned = a.copy()
bins = [0, 5, 8, 11, 15, 20, 30, 60, 210]
labels = ["0–5", "6–8", "9–11", "12–15", "16–20", "21–30", "31–60", "61+"]
a_binned["delivery_bin"] = pd.cut(a_binned["delivery_days"], bins=bins, labels=labels, include_lowest=True)
bin_stats = a_binned.groupby("delivery_bin", observed=True).agg(
    mean_score=("review_score", "mean"),
    median_score=("review_score", "median"),
    n=("review_score", "size"),
    top_box=("review_score", lambda s: (s == 5).mean() * 100),
    bottom_box=("review_score", lambda s: (s == 1).mean() * 100),
).reset_index()
print("\nDelivery-bin summary:")
print(bin_stats.to_string(index=False))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Delivery days by review score — ρ = {rho_a:+.3f}",
        "Top-box (5★) and bottom-box (1★) by delivery bin"
    ),
    horizontal_spacing=0.12,
)

# Left: box plot (delivery days distribution per star)
for star in [1, 2, 3, 4, 5]:
    fig.add_trace(
        go.Box(
            y=a.loc[a["review_score"] == star, "delivery_days"],
            name=f"{star}★",
            boxpoints=False,
            showlegend=False,
        ),
        row=1, col=1,
    )

# Right: top-box and bottom-box by delivery bin
fig.add_trace(
    go.Scatter(
        x=bin_stats["delivery_bin"].astype(str),
        y=bin_stats["top_box"],
        mode="lines+markers", name="Top-box (5★) %",
        line=dict(width=3, color="#2E8B57"),
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(
        x=bin_stats["delivery_bin"].astype(str),
        y=bin_stats["bottom_box"],
        mode="lines+markers", name="Bottom-box (1★) %",
        line=dict(width=3, color="#C0392B"),
    ),
    row=1, col=2,
)

fig.update_yaxes(title_text="Delivery days", row=1, col=1, range=[0, 60])
fig.update_xaxes(title_text="Review score", row=1, col=1)
fig.update_yaxes(title_text="Share of orders (%)", row=1, col=2)
fig.update_xaxes(title_text="Delivery days (binned)", row=1, col=2)
fig.update_layout(
    title_text="§4-A — Delivery time vs review score (delivered, n≈96k)",
    height=500, width=1150, template="plotly_white",
)

fig.show()
fig.write_image(PROJECT_ROOT / "reports" / "figures" / "08_4A_delivery_vs_review.png", scale=2)


Delivery-bin summary:
delivery_bin  mean_score  median_score     n   top_box  bottom_box
         0–5    4.447435           5.0 13374 69.859429    5.054584
         6–8    4.383040           5.0 20165 66.575750    5.623605
        9–11    4.323661           5.0 18300 63.808743    6.437158
       12–15    4.252990           5.0 17890 60.380101    7.098938
       16–20    4.127598           5.0 12077 55.179266    8.578289
       21–30    3.684919           4.0  9588 42.813934   17.136003
       31–60    2.260449           1.0  4139 16.501570   54.071032
         61+    2.140893           1.0   291 15.463918   60.824742


### §4-A — Result

**Test:** Spearman ρ on `delivery_days` vs `review_score`, delivered orders only.
**Sample:** n = 95,824 (654 rows dropped for missing review).

| Metric | Value |
|---|---|
| ρ | −0.235 |
| 95% CI | [−0.241, −0.228] |
| p-value | ≈ 0 |
| ρ² | 0.055 |
| Effect size label | weak (Cohen) |

**Headline: the 30-day cliff.** The ρ label understates the story. Top-box (5★)
and bottom-box (1★) rates are stable and favorable through ~20 delivery days
(top-box ≥55%, bottom-box ≤9% across the first five bins), then collapse across
two bins:

- **21–30 days (n=9,588):** top-box 42.8%, bottom-box 17.1% — first bin where
  the two curves cross.
- **31–60 days (n=4,139):** top-box **16.5%**, bottom-box **54.1%** — a −26pp
  top-box collapse and +37pp bottom-box surge in a single bin.

Past 30 days, more than half of orders receive 1★ and fewer than one in six
receive 5★. The relationship is threshold-shaped, not gradient-shaped, which is
why the Spearman ρ (−0.235, "weak" by convention) understates the operational
severity. Decision #11 (top-box/bottom-box as primary KPIs) was correct.

**Carry-forward.** The 30-day cliff motivates §4-B's binary framing: `on_time`
vs late is a cleaner operational lever than "deliver faster on average." Most
of the satisfaction damage lives in the tail past the promised date, not in
shaving two days off a typical on-time delivery.

In [44]:
# =========================================================
# §4-B: Does on-time delivery affect review score?
# Two framings: full ordinal (Mann-Whitney) + binary (bottom-box)
# =========================================================

# Delivered + reviewed + on_time known
b = delivered[["on_time", "review_score"]].dropna()
print(f"§4-B sample size: {len(b):,}")
print(f"  On-time: {b['on_time'].sum():,} ({b['on_time'].mean()*100:.2f}%)")
print(f"  Late:    {(~b['on_time']).sum():,} ({(~b['on_time']).mean()*100:.2f}%)")

# Group the scores
ontime_scores = b.loc[b["on_time"], "review_score"].values
late_scores   = b.loc[~b["on_time"], "review_score"].values

# ---------------------------------------------------------
# §4-B-1: Mann-Whitney U on full ordinal review_score
# ---------------------------------------------------------
print("\n" + "=" * 55)
print("§4-B-1: Mann-Whitney U (on_time vs late, full ordinal)")
print("=" * 55)

u_stat, u_pval = stats.mannwhitneyu(ontime_scores, late_scores, alternative="two-sided")
n1, n2 = len(ontime_scores), len(late_scores)

# Rank-biserial correlation = 1 - 2U/(n1*n2), from Mann-Whitney U
# Using scipy's U1 convention (U for first sample)
rb = 1 - (2 * u_stat) / (n1 * n2)

print(f"U statistic = {u_stat:,.0f}")
print(f"p-value     = {u_pval:.2e}")
print(f"Rank-biserial r = {rb:+.4f}")

# Descriptive: mean + top-box + bottom-box by group
for label, arr in [("On-time", ontime_scores), ("Late", late_scores)]:
    mean = arr.mean()
    top  = (arr == 5).mean() * 100
    bot  = (arr == 1).mean() * 100
    print(f"  {label:8s}: mean={mean:.2f}  top-box={top:5.1f}%  bottom-box={bot:5.1f}%")

# Cohen-style interpretation for rank-biserial (same thresholds as ρ)
abs_rb = abs(rb)
if   abs_rb < 0.10: rb_label = "trivial"
elif abs_rb < 0.30: rb_label = "weak"
elif abs_rb < 0.50: rb_label = "moderate"
else:               rb_label = "strong"
print(f"\nEffect size: {rb_label} (|r| = {abs_rb:.3f})")

§4-B sample size: 95,824
  On-time: 88,163 (92.01%)
  Late:    7,661 (7.99%)

§4-B-1: Mann-Whitney U (on_time vs late, full ordinal)
U statistic = 524,655,312
p-value     = 0.00e+00
Rank-biserial r = -0.5536
  On-time : mean=4.29  top-box= 62.4%  bottom-box=  6.6%
  Late    : mean=2.57  top-box= 22.2%  bottom-box= 46.2%

Effect size: strong (|r| = 0.554)


In [45]:
# ---------------------------------------------------------
# §4-B-2: Two-proportion z-test on bottom-box (1★) rate
# ---------------------------------------------------------
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

print("\n" + "=" * 55)
print("§4-B-2: Two-proportion z-test on bottom-box (1★) rate")
print("=" * 55)

bot_ontime = (ontime_scores == 1).sum()
bot_late   = (late_scores == 1).sum()

p_ontime = bot_ontime / n1
p_late   = bot_late   / n2

count   = np.array([bot_late, bot_ontime])
nobs    = np.array([n2, n1])

z_stat, z_pval = proportions_ztest(count, nobs, alternative="two-sided")

# 95% CI on each proportion (Wilson interval, better than normal approx near boundaries)
ci_ontime = proportion_confint(bot_ontime, n1, alpha=0.05, method="wilson")
ci_late   = proportion_confint(bot_late,   n2, alpha=0.05, method="wilson")

# Absolute pp gap + 95% CI (normal approx on the difference)
diff = p_late - p_ontime
se_diff = np.sqrt(p_ontime * (1 - p_ontime) / n1 + p_late * (1 - p_late) / n2)
diff_ci_lo = diff - 1.96 * se_diff
diff_ci_hi = diff + 1.96 * se_diff

# Odds ratio + 95% CI (log-OR method)
odds_ontime = p_ontime / (1 - p_ontime)
odds_late   = p_late   / (1 - p_late)
OR = odds_late / odds_ontime

log_or_se = np.sqrt(1/bot_ontime + 1/(n1 - bot_ontime) + 1/bot_late + 1/(n2 - bot_late))
log_or    = np.log(OR)
OR_ci_lo  = np.exp(log_or - 1.96 * log_or_se)
OR_ci_hi  = np.exp(log_or + 1.96 * log_or_se)

print(f"\nBottom-box (1★) rates:")
print(f"  On-time: {p_ontime*100:5.2f}%  95% CI [{ci_ontime[0]*100:.2f}%, {ci_ontime[1]*100:.2f}%]  (n={n1:,})")
print(f"  Late:    {p_late*100:5.2f}%  95% CI [{ci_late[0]*100:.2f}%, {ci_late[1]*100:.2f}%]  (n={n2:,})")

print(f"\nAbsolute gap: +{diff*100:.2f}pp  95% CI [{diff_ci_lo*100:+.2f}pp, {diff_ci_hi*100:+.2f}pp]")
print(f"Odds ratio (late vs on-time): {OR:.2f}×  95% CI [{OR_ci_lo:.2f}, {OR_ci_hi:.2f}]")
print(f"\nz = {z_stat:.2f}, p = {z_pval:.2e}")


§4-B-2: Two-proportion z-test on bottom-box (1★) rate

Bottom-box (1★) rates:
  On-time:  6.59%  95% CI [6.43%, 6.76%]  (n=88,163)
  Late:    46.22%  95% CI [45.11%, 47.34%]  (n=7,661)

Absolute gap: +39.63pp  95% CI [+38.50pp, +40.76pp]
Odds ratio (late vs on-time): 12.18×  95% CI [11.56, 12.83]

z = 112.11, p = 0.00e+00


In [46]:
# =========================================================
# §4-B visualization: two panels
# Left: stacked review score distribution, on-time vs late
# Right: bottom-box rate comparison with gap annotation
# =========================================================

# Review score distribution per group
def score_dist(arr):
    return {s: (arr == s).mean() * 100 for s in [1, 2, 3, 4, 5]}

dist_ontime = score_dist(ontime_scores)
dist_late   = score_dist(late_scores)

star_colors = {
    1: "#C0392B",  # red
    2: "#E67E22",  # orange
    3: "#F1C40F",  # yellow
    4: "#7DCEA0",  # light green
    5: "#229954",  # dark green
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Review score distribution — |r| = 0.55 (strong)",
        f"Bottom-box (1★) rate — OR = 12.2× (CI [11.6, 12.8])",
    ),
    horizontal_spacing=0.18,
    column_widths=[0.6, 0.4],
)

# Left: stacked bars (100%-normalized)
groups = ["On-time", "Late"]
for star in [1, 2, 3, 4, 5]:
    values = [dist_ontime[star], dist_late[star]]
    fig.add_trace(
        go.Bar(
            x=groups,
            y=values,
            name=f"{star}★",
            marker_color=star_colors[star],
            text=[f"{v:.1f}%" for v in values],
            textposition="inside",
            textfont=dict(color="white", size=11),
        ),
        row=1, col=1,
    )

# Right: bottom-box comparison bars
fig.add_trace(
    go.Bar(
        x=["On-time", "Late"],
        y=[p_ontime * 100, p_late * 100],
        marker_color=["#229954", "#C0392B"],
        text=[f"{p_ontime*100:.1f}%", f"{p_late*100:.1f}%"],
        textposition="outside",
        textfont=dict(size=14),
        showlegend=False,
    ),
    row=1, col=2,
)

# Annotate the gap
fig.add_annotation(
    xref="x2", yref="y2",
    x=0.5, y=max(p_ontime, p_late) * 100 * 0.55,
    text=f"+{diff*100:.1f}pp gap<br>OR = {OR:.1f}×",
    showarrow=False,
    font=dict(size=13, color="black"),
    bgcolor="rgba(255,255,200,0.8)",
    bordercolor="black",
    borderwidth=1,
)

fig.update_layout(
    barmode="stack",
    title_text="§4-B — On-time delivery vs review score (delivered, n=95,824)",
    height=520, width=1150, template="plotly_white",
    legend=dict(title="Score", orientation="v", x=0.56, y=1.0),
)
fig.update_yaxes(title_text="Share of orders (%)", row=1, col=1, range=[0, 100])
fig.update_yaxes(title_text="Bottom-box rate (%)", row=1, col=2, range=[0, 55])

# Reset barmode on right panel only: use a separate trace group
# (In plotly, barmode applies globally — but with only one trace on right panel,
# the stack setting doesn't affect it visually. Leaving as-is is fine.)

fig.show()
fig.write_image(PROJECT_ROOT / "reports" / "figures" / "09_4B_ontime_vs_review.png", scale=2)

### §4-B — Result

**Test 1 — Mann-Whitney U (full ordinal review score, on-time vs late)**
- Sample: n = 95,824 (on-time 88,163 / 92.0%, late 7,661 / 8.0%)
- U = 524,655,312, p ≈ 0
- **Rank-biserial |r| = 0.554 (strong)**
- Mean: on-time 4.29 vs late 2.57 (Δ = −1.72 stars)

**Test 2 — Two-proportion z on bottom-box (1★) rate**
- On-time: 6.59% 1★  (95% CI [6.43%, 6.76%])
- Late:    46.22% 1★  (95% CI [45.11%, 47.34%])
- **Absolute gap: +39.6pp  (95% CI [+38.5, +40.8])**
- **Odds ratio: 12.18× (95% CI [11.56, 12.83])**
- z = 112.1, p ≈ 0

**Interpretation.** Late delivery makes a 1★ review roughly twelve times more
likely in odds terms, or +40pp in absolute terms. Only 8% of orders arrive
late, but that 8% accounts for a hugely disproportionate share of the 1★
reviews — a textbook Pareto effect on the complaint distribution.

The ordinal distribution reveals the full inversion: on-time orders are 83%
4★-or-5★ and only 6.6% 1★; late orders flip this to 46% 1★ and only 35%
4★-or-5★. A late delivery doesn't just "lower" the review — it substantively
changes what review a customer is *going to leave*.

**Why §4-A labeled "weak" and §4-B labels "strong" for the same phenomenon.**
§4-A used continuous `delivery_days` across the full distribution, where 92% of
orders sit in the on-time mass with flat satisfaction — this dilutes any rank
correlation. §4-B isolates the binary split at the promised-date threshold,
which is precisely where the satisfaction cliff lives. The underlying data is
identical; the effect-size label depends entirely on which summary statistic
captures the threshold shape. This is why Phase 2 decision #11 (top-box and
bottom-box rates as primary KPIs) was correct: conventional ordinal-correlation
labels structurally understate threshold effects.

**Operational read.** Moving an order from late to on-time has a far larger
expected satisfaction return than shaving two days off an already-on-time
delivery. The lever is `on_time %`, not average delivery speed.

In [47]:
# =========================================================
# Q2: Does order price correlate with review score?
# Spearman rank correlation, delivered + reviewed only
# =========================================================

q2 = delivered[["items_price_total", "review_score"]].dropna()
print(f"Q2 sample size: {len(q2):,} (dropped {len(delivered) - len(q2)} rows)")

# Spearman
rho_q2, pval_q2 = stats.spearmanr(q2["items_price_total"], q2["review_score"])
print(f"\nSpearman ρ = {rho_q2:+.4f}")
print(f"p-value    = {pval_q2:.2e}")

# Bootstrap 95% CI
rng = np.random.default_rng(42)
boot_q2 = stats.bootstrap(
    (q2["items_price_total"].values, q2["review_score"].values),
    statistic=spearman_stat,
    n_resamples=2000,
    paired=True,
    confidence_level=0.95,
    random_state=rng,
    method="percentile",
)
ci_lo_q2, ci_hi_q2 = boot_q2.confidence_interval
print(f"95% CI     = [{ci_lo_q2:+.4f}, {ci_hi_q2:+.4f}]  (bootstrap, 2000 resamples)")

# Effect-size label
abs_rho_q2 = abs(rho_q2)
if   abs_rho_q2 < 0.10: label_q2 = "trivial"
elif abs_rho_q2 < 0.30: label_q2 = "weak"
elif abs_rho_q2 < 0.50: label_q2 = "moderate"
else:                   label_q2 = "strong"
print(f"\nEffect size: {label_q2} ({rho_q2:+.3f})")
print(f"ρ² = {rho_q2**2:.4f} ({rho_q2**2 * 100:.2f}% of rank-variance shared)")

Q2 sample size: 95,832 (dropped 646 rows)

Spearman ρ = -0.0288
p-value    = 5.03e-19
95% CI     = [-0.0352, -0.0223]  (bootstrap, 2000 resamples)

Effect size: trivial (-0.029)
ρ² = 0.0008 (0.08% of rank-variance shared)


In [48]:
# =========================================================
# Q2 visualization: flatness across the price range
# =========================================================

# Quintile for left panel (5 bars more readable than 10)
q2_q = q2.copy()
q2_q["price_quintile"] = pd.qcut(q2_q["items_price_total"], 5, labels=False) + 1
q2_q["price_decile"]   = pd.qcut(q2_q["items_price_total"], 10, labels=False) + 1

# Left panel data: review distribution per quintile (% stacked)
quintile_dist = (
    q2_q.groupby("price_quintile")["review_score"]
        .value_counts(normalize=True)
        .mul(100)
        .unstack(fill_value=0)
        .sort_index()
)

# Quintile labels with price range
quintile_ranges = q2_q.groupby("price_quintile")["items_price_total"].agg(["min", "max"]).round(0)
quintile_labels = [
    f"Q{i}<br>R${int(r['min'])}–{int(r['max'])}"
    for i, r in quintile_ranges.iterrows()
]

# Right panel data: decile top-box and bottom-box rates
decile_stats_q2 = q2_q.groupby("price_decile").agg(
    price_median=("items_price_total", "median"),
    top_box=("review_score", lambda s: (s == 5).mean() * 100),
    bottom_box=("review_score", lambda s: (s == 1).mean() * 100),
).reset_index()

overall_top = (q2["review_score"] == 5).mean() * 100
overall_bot = (q2["review_score"] == 1).mean() * 100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Review score distribution by price quintile — ρ = {rho_q2:+.3f}",
        "Top-box and bottom-box rates by price decile",
    ),
    horizontal_spacing=0.15,
)

# Left: stacked bars per quintile
for star in [1, 2, 3, 4, 5]:
    values = quintile_dist[star].values if star in quintile_dist.columns else [0]*5
    fig.add_trace(
        go.Bar(
            x=quintile_labels,
            y=values,
            name=f"{star}★",
            marker_color=star_colors[star],
            text=[f"{v:.0f}%" for v in values],
            textposition="inside",
            textfont=dict(color="white", size=10),
            showlegend=True,
        ),
        row=1, col=1,
    )

# Right: top-box and bottom-box lines + reference lines
fig.add_trace(
    go.Scatter(
        x=decile_stats_q2["price_decile"],
        y=decile_stats_q2["top_box"],
        mode="lines+markers", name="Top-box (5★) %",
        line=dict(width=3, color="#229954"),
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(
        x=decile_stats_q2["price_decile"],
        y=decile_stats_q2["bottom_box"],
        mode="lines+markers", name="Bottom-box (1★) %",
        line=dict(width=3, color="#C0392B"),
    ),
    row=1, col=2,
)
# Reference lines for overall averages
fig.add_hline(
    y=overall_top, line_dash="dash", line_color="#229954", opacity=0.5,
    annotation_text=f"Overall 5★: {overall_top:.1f}%",
    annotation_position="top left",
    row=1, col=2,
)
fig.add_hline(
    y=overall_bot, line_dash="dash", line_color="#C0392B", opacity=0.5,
    annotation_text=f"Overall 1★: {overall_bot:.1f}%",
    annotation_position="bottom left",
    row=1, col=2,
)

fig.update_layout(
    barmode="stack",
    title_text="Q2 — Price vs review score (delivered + reviewed, n≈96k)",
    height=520, width=1200, template="plotly_white",
    legend=dict(title="Score", orientation="v", x=0.46, y=1.0),
)
fig.update_yaxes(title_text="Share of orders (%)", row=1, col=1, range=[0, 100])
fig.update_yaxes(title_text="Rate (%)", row=1, col=2, range=[0, 80])
fig.update_xaxes(title_text="Price quintile", row=1, col=1)
fig.update_xaxes(title_text="Price decile (1 = cheapest, 10 = most expensive)", row=1, col=2)

fig.show()
fig.write_image(PROJECT_ROOT / "reports" / "figures" / "10_q2_price_vs_review.png", scale=2)

### Q2 — Result

**Test:** Spearman ρ on `items_price_total` vs `review_score`, delivered + reviewed only.
**Sample:** n = 95,832 (646 rows dropped for missing review).

| Metric | Value |
|---|---|
| ρ | −0.029 |
| 95% CI | [−0.035, −0.022] |
| p-value | 5.0 × 10⁻¹⁹ |
| ρ² | 0.0008 |
| Effect size | trivial |

**Interpretation.** Price is functionally uncorrelated with satisfaction. The
95% CI rules out even "weak" effects (|ρ| < 0.04 across the range), and
ρ² = 0.08% means price explains less than one-tenth of one percent of
review-score rank variance. The p-value is "significant" only because n is
large enough to detect any ρ ≠ 0.

The visual confirms: review-score distribution is essentially identical across
all five price quintiles (top-box 58–60%, bottom-box 8–12%) despite the Q5
price range being 30× the Q1 range. A small bottom-box uptick in the highest
decile (~13% vs 9.8% baseline) is the only visible wiggle, and it is
operationally trivial.

**Mediation note.** The tiny observed correlation is consistent with full
mediation through delivery time. Under a naive decomposition:
- Q1: price → delivery ρ = +0.102
- §4-A: delivery → review ρ = −0.235
- Indirect path: +0.102 × (−0.235) = **−0.024**
- Observed Q2 ρ: **−0.029**

These match to within noise, suggesting the small price→review signal is
almost entirely the echo of price's effect on delivery speed, not a direct
"expensive items disappoint" pathway. This is an observational coincidence
of magnitudes rather than a formal mediation test, but it is a strikingly
clean pattern.

**Portfolio finding (confirms #4).** Price does not drive satisfaction.
Delivery does. The §3 pattern from Phase 2 is validated at effect-size
precision.

**Carry-forward.** No price-conditioning needed anywhere downstream.

In [49]:
# =========================================================
# Q3: Does product category affect review score?
# Kruskal-Wallis + Dunn's post-hoc, delivered + reviewed only
# =========================================================

q3 = delivered[["category", "review_score"]].dropna()
q3 = q3[q3["category"].notna() & (q3["category"] != "")]
print(f"Q3 sample size (pre-filter): {len(q3):,}")
print(f"Unique categories (raw):     {q3['category'].nunique()}")

# Minimum-n filter: categories must have ≥100 orders for rank-stats reliability
MIN_N = 100
cat_counts = q3["category"].value_counts()
kept_cats = cat_counts[cat_counts >= MIN_N].index
dropped_cats = cat_counts[cat_counts < MIN_N]

q3_filtered = q3[q3["category"].isin(kept_cats)].copy()
print(f"\nAfter n ≥ {MIN_N} filter:")
print(f"  Categories kept:    {len(kept_cats)}")
print(f"  Categories dropped: {len(dropped_cats)} (n < {MIN_N}, total orders: {dropped_cats.sum()})")
print(f"  Sample size used:   {len(q3_filtered):,}")

# Kruskal-Wallis
groups = [q3_filtered.loc[q3_filtered["category"] == c, "review_score"].values
          for c in kept_cats]
h_stat, h_pval = stats.kruskal(*groups)

k = len(kept_cats)
n = len(q3_filtered)
eta_sq_h = (h_stat - k + 1) / (n - k)

print(f"\nKruskal-Wallis:")
print(f"  H statistic = {h_stat:,.2f}")
print(f"  df          = {k - 1}")
print(f"  p-value     = {h_pval:.2e}")
print(f"  η²_H        = {eta_sq_h:.4f}")

if   eta_sq_h < 0.01: eta_label = "trivial"
elif eta_sq_h < 0.06: eta_label = "small"
elif eta_sq_h < 0.14: eta_label = "moderate"
else:                 eta_label = "large"
print(f"  Effect size: {eta_label}")

# Per-category summary
cat_summary = q3_filtered.groupby("category").agg(
    n=("review_score", "size"),
    mean=("review_score", "mean"),
    median=("review_score", "median"),
    top_box=("review_score", lambda s: (s == 5).mean() * 100),
    bottom_box=("review_score", lambda s: (s == 1).mean() * 100),
).round(3).sort_values("mean", ascending=False)

print(f"\nRange of category means: {cat_summary['mean'].min():.2f} to {cat_summary['mean'].max():.2f}")
print(f"Range width:             {cat_summary['mean'].max() - cat_summary['mean'].min():.2f} stars")

print("\nTop 5 categories (highest mean):")
print(cat_summary.head().to_string())
print("\nBottom 5 categories (lowest mean):")
print(cat_summary.tail().to_string())

Q3 sample size (pre-filter): 94,463
Unique categories (raw):     71

After n ≥ 100 filter:
  Categories kept:    51
  Categories dropped: 20 (n < 100, total orders: 798)
  Sample size used:   93,665

Kruskal-Wallis:
  H statistic = 789.81
  df          = 50
  p-value     = 1.11e-133
  η²_H        = 0.0079
  Effect size: trivial

Range of category means: 3.65 to 4.54
Range width:             0.90 stars

Top 5 categories (highest mean):
                           n   mean  median  top_box  bottom_box
category                                                        
books_general_interest   489  4.542     5.0   75.051       5.112
food_drink               219  4.447     5.0   66.210       2.740
books_technical          253  4.423     5.0   71.542       6.719
luggage_accessories     1008  4.371     5.0   65.972       5.754
food                     432  4.336     5.0   65.972       7.639

Bottom 5 categories (lowest mean):
                          n   mean  median  top_box  bottom_box
catego

In [50]:
# ---------------------------------------------------------
# Q3: Dunn's post-hoc with Bonferroni correction
# ---------------------------------------------------------
# scikit-posthocs is the standard implementation; install if missing
try:
    import scikit_posthocs as sp
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-posthocs"])
    import scikit_posthocs as sp

dunn = sp.posthoc_dunn(
    q3_filtered, val_col="review_score", group_col="category",
    p_adjust="bonferroni",
)
# dunn is a square DataFrame of adjusted p-values; diagonal is 1.0

# Count significant pairs (p < 0.05), upper triangle only to avoid double-count
mask = np.triu(np.ones_like(dunn, dtype=bool), k=1)
pvals_upper = dunn.values[mask]
total_pairs = len(pvals_upper)
sig_pairs   = (pvals_upper < 0.05).sum()

print(f"Dunn's test (Bonferroni-corrected):")
print(f"  Total pairs tested:      {total_pairs:,}")
print(f"  Significant (p < 0.05):  {sig_pairs:,} ({sig_pairs / total_pairs * 100:.1f}%)")

# Top 5 largest mean gaps among significant pairs
sig_gaps = []
cats_list = list(dunn.index)
for i, c1 in enumerate(cats_list):
    for j, c2 in enumerate(cats_list):
        if j <= i:
            continue
        p_adj = dunn.iloc[i, j]
        if p_adj < 0.05:
            gap = abs(cat_summary.loc[c1, "mean"] - cat_summary.loc[c2, "mean"])
            sig_gaps.append((c1, c2, gap, p_adj))

sig_gaps_df = pd.DataFrame(sig_gaps, columns=["cat_1", "cat_2", "mean_gap", "p_adj"])
sig_gaps_df = sig_gaps_df.sort_values("mean_gap", ascending=False)

print(f"\nTop 5 largest significant mean-score gaps:")
print(sig_gaps_df.head().to_string(index=False))

Dunn's test (Bonferroni-corrected):
  Total pairs tested:      1,275
  Significant (p < 0.05):  186 (14.6%)

Top 5 largest significant mean-score gaps:
                 cat_1                 cat_2  mean_gap        p_adj
books_general_interest      office_furniture     0.896 6.963739e-42
            food_drink      office_furniture     0.801 4.366583e-13
       books_technical      office_furniture     0.777 5.212706e-19
   luggage_accessories      office_furniture     0.725 6.905091e-39
books_general_interest fashion_male_clothing     0.723 3.500838e-03


In [51]:
# =========================================================
# Q3 visualization: category ranking + spread
# Left: horizontal lollipop of top 10 + bottom 10 category means
# Right: scatter of category mean vs n, colored by top-box
# =========================================================
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Category mean review — top 10 & bottom 10 of {k} categories (overall mean = {overall_mean:.2f})",
        "All categories: mean review vs order volume",
    ),
    horizontal_spacing=0.22,
    column_widths=[0.55, 0.45],
)

# Left: horizontal lollipop
for _, row in lollipop.iterrows():
    color = "#229954" if row["rank_label"] == "Top 10" else "#C0392B"
    fig.add_trace(
        go.Scatter(
            x=[overall_mean, row["mean"]],
            y=[row["category"], row["category"]],
            mode="lines",
            line=dict(color=color, width=2),
            showlegend=False,
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[row["mean"]],
            y=[row["category"]],
            mode="markers+text",
            marker=dict(size=10, color=color),
            text=[f"{row['mean']:.2f}"],
            textposition="middle right",
            textfont=dict(size=10),
            showlegend=False,
        ),
        row=1, col=1,
    )

# Overall mean reference WITHOUT annotation (info moved into subtitle)
fig.add_vline(
    x=overall_mean, line_dash="dash", line_color="gray",
    row=1, col=1,
)

# Right: scatter of category mean vs n
cat_plot = cat_summary.reset_index()
fig.add_trace(
    go.Scatter(
        x=cat_plot["n"], y=cat_plot["mean"],
        mode="markers",
        marker=dict(
            size=9,
            color=cat_plot["top_box"],
            colorscale="RdYlGn",
            cmin=40, cmax=70,
            colorbar=dict(title="Top-box %", x=1.02, thickness=15),
            line=dict(width=0.5, color="black"),
        ),
        text=cat_plot["category"],
        hovertemplate="<b>%{text}</b><br>n=%{x:,}<br>mean=%{y:.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=2,
)
fig.add_hline(y=overall_mean, line_dash="dash", line_color="gray", row=1, col=2)

# Label office_furniture outlier explicitly on the scatter
of_row = cat_plot[cat_plot["category"] == "office_furniture"].iloc[0]
fig.add_annotation(
    x=np.log10(of_row["n"]), y=of_row["mean"],
    xref="x2", yref="y2",
    text="office_furniture",
    showarrow=True, arrowhead=2, ax=-40, ay=-30,
    font=dict(size=10, color="#8B0000"),
)

fig.update_xaxes(title_text="Mean review score", row=1, col=1,
                 range=[max(1, lollipop["mean"].min() - 0.3), 5.1])
fig.update_yaxes(title_text="", row=1, col=1, tickfont=dict(size=9))
fig.update_xaxes(type="log", title_text="Order volume (log)", row=1, col=2)
fig.update_yaxes(title_text="Category mean review", row=1, col=2)

fig.update_layout(
    title_text=(
        f"Q3 — Category vs review score  |  "
        f"H = {h_stat:.0f}, η²_H = {eta_sq_h:.3f} ({eta_label})  |  "
        f"mean range = {cat_summary['mean'].max() - cat_summary['mean'].min():.2f} stars"
    ),
    height=650, width=1250, template="plotly_white",
    margin=dict(l=180, t=110),
)

fig.show()
fig.write_image(PROJECT_ROOT / "reports" / "figures" / "11_q3_category_vs_review.png", scale=2)

### Q3 — Result

**Test 1 — Kruskal-Wallis (category → review_score)**
- Sample: n = 93,665 delivered + reviewed orders across 51 categories
  (20 categories with n < 100 dropped: 798 orders, 0.8% of pre-filter sample)
- H = 789.81, df = 50, p = 1.1 × 10⁻¹³³
- **η²_H = 0.0079 — trivial (Cohen threshold: <0.01 trivial)**

**Test 2 — Dunn's post-hoc (Bonferroni-corrected)**
- Total pairs: 1,275
- **Significant pairs (p_adj < 0.05): 186 (14.6%)**
- 85% of category pairs are statistically indistinguishable after correction

**Range of category means:** 3.65 (office_furniture) to 4.54 (books_general_interest),
width = 0.90 stars across the full 51-category span.

**Top 5 categories by mean:** books_general_interest (4.54), food_drink (4.45),
books_technical (4.42), luggage_accessories (4.37), food (4.34).

**Bottom 5 categories by mean:** fixed_telephony (3.97), home_confort (3.91),
audio (3.85), fashion_male_clothing (3.82), office_furniture (3.65).

**Interpretation.** Product category has a statistically detectable but
operationally trivial effect on review score. The 0.90-star max range sounds
substantial, but η²_H = 0.008 means category explains less than 1% of
review-score rank variance — and only 15% of category pairs differ
significantly after Bonferroni correction. The large majority of the Olist
catalog behaves identically to the average on satisfaction.

**One real operational signal: office_furniture.** At n = 1,241, office_furniture
has a mean of 3.65 and bottom-box rate of 17.2% — noticeably worse than the
next-worst category and sitting ~0.5 stars below the overall mean. This is
the outlier on the right-panel scatter. Plausible mechanisms (damage in
transit, assembly complexity, size-expectation mismatch) cannot be confirmed
from this data but would be the first place to look in a retention
investigation. audio (n=341, mean 3.85) and fashion_male_clothing (n=105,
mean 3.82) show a similar but smaller pattern.

**Portfolio finding (confirms #4, reinforces #5).** Category is not a
meaningful driver of satisfaction. The 30-day cliff (§4-A) and on_time
effect (§4-B) remain the real levers. This is the third consecutive test
where a non-delivery variable produced a significant-but-trivial result,
cementing the central thesis: **Olist satisfaction is delivery-driven, not
product-driven.**

**Methodological note.** Minimum-n filter (≥100) excluded 20 tiny categories
whose rank statistics would have been noisy. Trade-off: clean test vs
completeness of coverage. Since dropped categories represent 0.8% of
post-delivered sample, the filter has negligible impact on external
validity.

In [52]:
# =========================================================
# Q4a: Does payment method affect order completion?
# Chi-square on payment_type × completed_vs_not (all statuses)
# =========================================================

# All statuses (decision #2), payment_type known
q4 = orders[orders["payment_type"].notna() & (orders["payment_type"] != "")].copy()
q4["completed"] = (q4["order_status"] == "delivered")

# Drop tiny cells: payment_type 'not_defined' or similar edge cases (if any)
pt_counts = q4["payment_type"].value_counts()
print("Payment type counts (all statuses):")
print(pt_counts)
print(f"\nCompletion rate overall: {q4['completed'].mean()*100:.2f}%")

# Keep only payment types with n ≥ 50 (same spirit as Q3's min-n filter)
valid_pt = pt_counts[pt_counts >= 50].index
q4 = q4[q4["payment_type"].isin(valid_pt)].copy()
print(f"\nPayment types kept: {list(valid_pt)}")
print(f"Sample size used:   {len(q4):,}")

# 4×2 contingency table
contingency = pd.crosstab(q4["payment_type"], q4["completed"])
contingency.columns = ["not_completed", "completed"]
contingency["total"] = contingency.sum(axis=1)
contingency["completion_rate_pct"] = (contingency["completed"] / contingency["total"] * 100).round(3)

print("\nContingency table:")
print(contingency.to_string())

# Chi-square test (4×2)
chi2_stat, chi2_pval, chi2_dof, expected = stats.chi2_contingency(
    contingency[["not_completed", "completed"]].values
)
n_total = contingency[["not_completed", "completed"]].values.sum()

# Cramer's V for 4×2 (phi is only for 2×2)
min_dim = min(contingency.shape[0], 2) - 1
cramer_v = np.sqrt(chi2_stat / (n_total * min_dim))

print(f"\n4×2 Chi-square:")
print(f"  χ² = {chi2_stat:.2f}")
print(f"  df = {chi2_dof}")
print(f"  p  = {chi2_pval:.4f}")
print(f"  Cramer's V = {cramer_v:.4f}")

if   cramer_v < 0.10: v_label = "trivial"
elif cramer_v < 0.30: v_label = "weak"
elif cramer_v < 0.50: v_label = "moderate"
else:                 v_label = "strong"
print(f"  Effect size: {v_label}")

# Follow-up: 2×2 credit_card vs boleto (original plan)
print("\n" + "=" * 55)
print("Follow-up: 2×2 Chi-square (credit_card vs boleto)")
print("=" * 55)
cb = q4[q4["payment_type"].isin(["credit_card", "boleto"])]
cb_cont = pd.crosstab(cb["payment_type"], cb["completed"])

chi2_cb, p_cb, dof_cb, _ = stats.chi2_contingency(cb_cont.values)
phi_cb = np.sqrt(chi2_cb / cb_cont.values.sum())  # phi is valid for 2×2

print(cb_cont.to_string())
print(f"\nχ² = {chi2_cb:.2f}, df = {dof_cb}, p = {p_cb:.4f}")
print(f"φ (phi) = {phi_cb:.4f}")

Payment type counts (all statuses):
payment_type
credit_card    76504
boleto         19784
voucher         1621
debit_card      1528
not_defined        3
Name: count, dtype: int64

Completion rate overall: 97.02%

Payment types kept: ['credit_card', 'boleto', 'voucher', 'debit_card']
Sample size used:   99,437

Contingency table:
              not_completed  completed  total  completion_rate_pct
payment_type                                                      
boleto                  593      19191  19784               97.003
credit_card            2201      74303  76504               97.123
debit_card               43       1485   1528               97.186
voucher                 123       1498   1621               92.412

4×2 Chi-square:
  χ² = 122.15
  df = 3
  p  = 0.0000
  Cramer's V = 0.0350
  Effect size: trivial

Follow-up: 2×2 Chi-square (credit_card vs boleto)
completed     False  True 
payment_type              
boleto          593  19191
credit_card    2201  74303

χ² = 0.

In [53]:
# =========================================================
# Q4b: Does payment method affect review score?
# Kruskal-Wallis across payment types, delivered + reviewed only
# =========================================================

q4b = delivered[["payment_type", "review_score"]].dropna()
q4b = q4b[q4b["payment_type"].isin(valid_pt)].copy()
print(f"Q4b sample size: {len(q4b):,}")

# Per-type summary
pt_summary = q4b.groupby("payment_type").agg(
    n=("review_score", "size"),
    mean=("review_score", "mean"),
    median=("review_score", "median"),
    top_box=("review_score", lambda s: (s == 5).mean() * 100),
    bottom_box=("review_score", lambda s: (s == 1).mean() * 100),
).round(3).sort_values("mean", ascending=False)
print("\nPer-payment-type review summary:")
print(pt_summary.to_string())
print(f"\nRange of means: {pt_summary['mean'].max() - pt_summary['mean'].min():.3f} stars")

# Kruskal-Wallis
groups_q4b = [q4b.loc[q4b["payment_type"] == p, "review_score"].values
              for p in valid_pt]
h_q4b, p_q4b = stats.kruskal(*groups_q4b)

k_q4b = len(valid_pt)
n_q4b = len(q4b)
eta_sq_q4b = (h_q4b - k_q4b + 1) / (n_q4b - k_q4b)

print(f"\nKruskal-Wallis (4 payment types):")
print(f"  H = {h_q4b:.2f}, df = {k_q4b - 1}, p = {p_q4b:.2e}")
print(f"  η²_H = {eta_sq_q4b:.4f}")

if   eta_sq_q4b < 0.01: eta_label_q4b = "trivial"
elif eta_sq_q4b < 0.06: eta_label_q4b = "small"
elif eta_sq_q4b < 0.14: eta_label_q4b = "moderate"
else:                   eta_label_q4b = "large"
print(f"  Effect size: {eta_label_q4b}")

Q4b sample size: 95,831

Per-payment-type review summary:
                  n   mean  median  top_box  bottom_box
payment_type                                           
debit_card     1478  4.241     5.0   62.043       8.051
boleto        19062  4.156     5.0   58.829       9.543
credit_card   73808  4.155     5.0   59.269       9.834
voucher        1483  4.105     5.0   59.002      10.519

Range of means: 0.136 stars

Kruskal-Wallis (4 payment types):
  H = 7.40, df = 3, p = 6.02e-02
  η²_H = 0.0000
  Effect size: trivial


In [54]:
# =========================================================
# Q4 combined visualization: completion + review across payment types
# =========================================================

overall_completion = q4["completed"].mean() * 100
overall_review_mean = q4b["review_score"].mean()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Q4a — Completion rate by payment type (Cramer's V = {cramer_v:.3f})",
        f"Q4b — Review score distribution by payment type (η²_H = {eta_sq_q4b:.3f})",
    ),
    horizontal_spacing=0.16,
    column_widths=[0.4, 0.6],
)

# Left: completion rate bars
completion_sorted = contingency.sort_values("completion_rate_pct", ascending=False)
fig.add_trace(
    go.Bar(
        x=completion_sorted.index,
        y=completion_sorted["completion_rate_pct"],
        marker_color="#5B9BD5",
        text=[f"{v:.2f}%<br>(n={int(completion_sorted.loc[i, 'total']):,})"
              for i, v in zip(completion_sorted.index, completion_sorted["completion_rate_pct"])],
        textposition="outside",
        textfont=dict(size=11),
        showlegend=False,
    ),
    row=1, col=1,
)
fig.add_hline(
    y=overall_completion, line_dash="dash", line_color="gray",
    annotation_text=f"Overall: {overall_completion:.2f}%",
    annotation_position="bottom right",
    row=1, col=1,
)

# Right: stacked review score distribution (100%-normalized)
pt_order = pt_summary.index.tolist()
review_dist = (
    q4b.groupby("payment_type")["review_score"]
       .value_counts(normalize=True)
       .mul(100)
       .unstack(fill_value=0)
       .loc[pt_order]
       .sort_index(axis=1)
)

for star in [1, 2, 3, 4, 5]:
    if star not in review_dist.columns:
        continue
    values = review_dist[star].values
    fig.add_trace(
        go.Bar(
            x=pt_order,
            y=values,
            name=f"{star}★",
            marker_color=star_colors[star],
            text=[f"{v:.1f}%" for v in values],
            textposition="inside",
            textfont=dict(color="white", size=10),
        ),
        row=1, col=2,
    )

fig.update_layout(
    barmode="stack",
    title_text=(
        f"Q4 — Payment method vs completion & review  |  "
        f"both effects trivial  |  n_completion={len(q4):,}, n_review={len(q4b):,}"
    ),
    height=520, width=1200, template="plotly_white",
    legend=dict(title="Score", orientation="v", x=1.02, y=1.0),
)
fig.update_yaxes(title_text="Completion rate (%)", row=1, col=1,
                 range=[90, 100])
fig.update_yaxes(title_text="Share of orders (%)", row=1, col=2, range=[0, 100])
fig.update_xaxes(title_text="Payment type", row=1, col=1)
fig.update_xaxes(title_text="Payment type", row=1, col=2)

fig.show()
fig.write_image(PROJECT_ROOT / "reports" / "figures" / "12_q4_payment_vs_completion_and_review.png", scale=2)

### Q4 — Result

#### Q4a — Payment method → order completion

**Test 1 — Full 4×2 Chi-square (all four payment types)**
- Sample: n = 99,437 (all statuses, decision #2)
- χ² = 122.15, df = 3, p < 10⁻²⁵
- **Cramer's V = 0.035 — trivial**

**Test 2 — 2×2 Chi-square (credit_card vs boleto, original plan)**
- χ² = 0.77, df = 1, p = 0.381
- **φ = 0.003 — trivial**

**Completion rates by payment type:**
| Payment type | n | Completion rate |
|---|---|---|
| debit_card | 1,528 | 97.19% |
| credit_card | 76,504 | 97.12% |
| boleto | 19,784 | 97.00% |
| **voucher** | **1,621** | **92.41%** |

**Interpretation.** The headline χ² = 122 is statistically significant but
operationally trivial at Cramer's V = 0.035. The credit/boleto/debit cluster
sits within 0.2pp of each other — textbook null, confirmed by the follow-up
2×2 credit-vs-boleto test (φ = 0.003, p = 0.381).

**The voucher exception.** At 92.41% completion, voucher orders complete
roughly 4.7pp less often than the other three methods and drive nearly all
of the 4×2 χ² signal. The gap is real but operationally narrow: only 1.6%
of orders pay by voucher, so the category cannot move the overall completion
rate meaningfully. Plausible mechanism: voucher orders likely skew toward
promotional, low-commitment, or gift-flow purchases where cancellation is
easier and buyer intent is weaker than credit-card checkout. This is a
customer-intent composition finding, not a payment-infrastructure one.

#### Q4b — Payment method → review score

**Test — Kruskal-Wallis (4 payment types on review_score)**
- Sample: n = 95,831 delivered + reviewed
- H = 7.40, df = 3, p = 0.060
- **η²_H ≈ 0.000 — trivial**

**Range of means:** 0.136 stars (debit_card 4.24 to voucher 4.11).

**Interpretation.** Payment method does not affect satisfaction. This is the
only test in Phase 4 where p exceeds 0.05 — at n = 95,831, an effect that
cannot achieve conventional significance is not merely "trivial," it is
effectively zero. The 0.136-star range across four methods is noise.

**Portfolio finding (confirms #4).** A fourth consecutive test where a
non-delivery variable produces no meaningful effect on satisfaction. With
Q1 (price→delivery, trivial), Q2 (price→review, trivial), Q3 (category→review,
trivial with one narrow outlier), and now Q4a/b (payment→everything, trivial
with one composition exception), the central thesis is now thoroughly
validated: **Olist satisfaction is delivery-driven. Nothing else materially
moves the needle.**

**Methodological note.** Following Q3's convention, the "not_defined"
payment type (n=3) was dropped. All four retained types have n ≥ 50.

In [55]:
# =========================================================
# §1: Does product category explain price?
# Welch's ANOVA on log10(items_price_total), Games-Howell post-hoc
# =========================================================

# pingouin has clean Welch's ANOVA + Games-Howell; install if missing
try:
    import pingouin as pg
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pingouin"])
    import pingouin as pg

# Data: all orders, valid price, valid category, ≥100 category filter (reuse kept_cats from Q3)
s1 = orders[["category", "items_price_total"]].dropna()
s1 = s1[(s1["items_price_total"] > 0) & (s1["category"].isin(kept_cats))].copy()
s1["log10_price"] = np.log10(s1["items_price_total"])
print(f"§1 sample size: {len(s1):,}")
print(f"Categories:     {s1['category'].nunique()}")

# Welch's ANOVA
welch = pg.welch_anova(data=s1, dv="log10_price", between="category")
print("\nWelch's ANOVA:")
print(welch.to_string(index=False))

# Effect size: omega-squared
# ω² = (SS_between - df_between * MS_within) / (SS_total + MS_within)
# pingouin's welch_anova returns np² (partial eta-sq); compute ω² manually from standard anova
anova_classic = pg.anova(data=s1, dv="log10_price", between="category", detailed=True)
ss_between = anova_classic.loc[anova_classic["Source"] == "category", "SS"].values[0]
ss_within  = anova_classic.loc[anova_classic["Source"] == "Within", "SS"].values[0]
df_between = anova_classic.loc[anova_classic["Source"] == "category", "DF"].values[0]
df_within  = anova_classic.loc[anova_classic["Source"] == "Within", "DF"].values[0]
ms_within  = ss_within / df_within
ss_total   = ss_between + ss_within
omega_sq   = (ss_between - df_between * ms_within) / (ss_total + ms_within)

print(f"\nω² (omega-squared): {omega_sq:.4f}")
if   omega_sq < 0.01: omega_label = "trivial"
elif omega_sq < 0.06: omega_label = "small"
elif omega_sq < 0.14: omega_label = "moderate"
else:                 omega_label = "large"
print(f"Effect size: {omega_label}")

# Per-category log10_price summary (for visualization + write-up)
cat_price = s1.groupby("category").agg(
    n=("log10_price", "size"),
    mean_log=("log10_price", "mean"),
    median_price=("items_price_total", "median"),
    q25_price=("items_price_total", lambda s: s.quantile(0.25)),
    q75_price=("items_price_total", lambda s: s.quantile(0.75)),
).round(3).sort_values("mean_log", ascending=False)

print(f"\nMedian price range: R${cat_price['median_price'].min():.2f} to R${cat_price['median_price'].max():.2f}")
print(f"Ratio most/least expensive: {cat_price['median_price'].max() / cat_price['median_price'].min():.1f}×")

print("\nTop 5 categories by median price:")
print(cat_price.nlargest(5, "median_price")[["n", "median_price", "q25_price", "q75_price"]].to_string())
print("\nBottom 5 categories by median price:")
print(cat_price.nsmallest(5, "median_price")[["n", "median_price", "q25_price", "q75_price"]].to_string())

§1 sample size: 96,398
Categories:     51

Welch's ANOVA:
  Source  ddof1       ddof2          F  p_unc      np2
category     50 6554.406749 339.846947    0.0 0.129529

ω² (omega-squared): 0.1291
Effect size: moderate

Median price range: R$21.90 to R$1200.00
Ratio most/least expensive: 54.8×

Top 5 categories by median price:
                               n  median_price  q25_price  q75_price
category                                                            
computers                    181      1200.000    730.000   1437.000
agro_industry_and_commerce   182       344.900     58.035    439.998
home_appliances_2            234       229.990    159.000    589.600
construction_tools_lights    232       164.500     78.225    230.000
office_furniture            1270       163.475    126.142    229.990

Bottom 5 categories by median price:
                           n  median_price  q25_price  q75_price
category                                                        
electronics         

In [56]:
# ---------------------------------------------------------
# §1: Games-Howell post-hoc
# ---------------------------------------------------------
# Games-Howell controls family-wise error and doesn't assume equal variances
# WARNING: 51 categories = 1,275 pairwise comparisons. May take 30-90 seconds.

print("Running Games-Howell on 51 categories (1,275 pairs)... this will take a moment.")
gh = pg.pairwise_gameshowell(data=s1, dv="log10_price", between="category")
print(f"Comparisons returned: {len(gh)}")

# Significant after default FWE control
sig_mask = gh["pval"] < 0.05
print(f"Significant pairs (p < 0.05): {sig_mask.sum():,} of {len(gh):,} ({sig_mask.mean()*100:.1f}%)")

# Largest effect-size pairs
gh["abs_diff_log"] = gh["diff"].abs()
gh_sorted = gh.sort_values("abs_diff_log", ascending=False)

print("\nTop 5 largest log10-price gaps:")
print(gh_sorted[["A", "B", "diff", "abs_diff_log", "pval", "hedges"]].head().to_string(index=False))

# Translate log diff back to price ratio for interpretability
print("\nTop 5 in price-ratio terms (10^diff = how many times more/less expensive):")
for _, r in gh_sorted.head().iterrows():
    ratio = 10 ** r["abs_diff_log"]
    print(f"  {r['A']} vs {r['B']}: {ratio:.1f}× price difference")

Running Games-Howell on 51 categories (1,275 pairs)... this will take a moment.


c:\Users\thinkpad\Desktop\olist-ecommerce-analysis\venv\Lib\site-packages\scipy\integrate\_quadpack_py.py:1286: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  quad_r = quad(f, low, high, args=args, full_output=self.full_output,


Comparisons returned: 1275
Significant pairs (p < 0.05): 899 of 1,275 (70.5%)

Top 5 largest log10-price gaps:
                 A           B      diff  abs_diff_log         pval    hedges
         computers electronics  1.532268      1.532268 0.000000e+00  3.646009
         computers   telephony  1.427664      1.427664 0.000000e+00  3.527588
christmas_supplies   computers -1.385994      1.385994 0.000000e+00 -4.329514
         computers  food_drink  1.358492      1.358492 1.867395e-13  4.588811
         computers      drinks  1.332403      1.332403 2.171596e-13  4.450336

Top 5 in price-ratio terms (10^diff = how many times more/less expensive):
  computers vs electronics: 34.1× price difference
  computers vs telephony: 26.8× price difference
  christmas_supplies vs computers: 24.3× price difference
  computers vs food_drink: 22.8× price difference
  computers vs drinks: 21.5× price difference


In [57]:
# =========================================================
# §1 visualization: price distribution across categories
# Left: top 10 + bottom 10 by median price, box plots
# Right: horizontal scatter of median price vs n, category-sized dots
# =========================================================

# Select top 10 + bottom 10 by median price
top10_price = cat_price.nlargest(10, "median_price").index.tolist()
bot10_price = cat_price.nsmallest(10, "median_price").index.tolist()
selected_cats = bot10_price[::-1] + top10_price  # cheapest first for chart reading

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Median price — top 10 and bottom 10 of {len(kept_cats)} categories",
        "All categories: median price vs order volume",
    ),
    horizontal_spacing=0.22,
    column_widths=[0.55, 0.45],
)

# Left: box plots per category on log scale
for cat in selected_cats:
    data_c = s1.loc[s1["category"] == cat, "items_price_total"]
    color = "#C0392B" if cat in bot10_price else "#229954"
    fig.add_trace(
        go.Box(
            x=data_c, y=[cat] * len(data_c),
            name=cat, orientation="h",
            boxpoints=False,
            marker_color=color,
            showlegend=False,
        ),
        row=1, col=1,
    )

# Right: scatter of median price vs n
cat_plot_price = cat_price.reset_index()
fig.add_trace(
    go.Scatter(
        x=cat_plot_price["n"],
        y=cat_plot_price["median_price"],
        mode="markers",
        marker=dict(
            size=10,
            color=cat_plot_price["median_log"] if "median_log" in cat_plot_price.columns else cat_plot_price["mean_log"],
            colorscale="Viridis",
            colorbar=dict(title="Mean log₁₀(price)", x=1.02, thickness=15),
            line=dict(width=0.5, color="black"),
        ),
        text=cat_plot_price["category"],
        hovertemplate="<b>%{text}</b><br>n=%{x:,}<br>median=R$%{y:.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=2,
)

fig.update_xaxes(type="log", title_text="Price (BRL, log)", row=1, col=1)
fig.update_yaxes(title_text="", row=1, col=1, tickfont=dict(size=9))
fig.update_xaxes(type="log", title_text="Order volume (log)", row=1, col=2)
fig.update_yaxes(type="log", title_text="Median price (BRL, log)", row=1, col=2)

fig.update_layout(
    title_text=(
        f"§1 — Category vs price  |  "
        f"Welch's F = {welch['F'].values[0]:.0f}, ω² = {omega_sq:.2f} ({omega_label})  |  "
        f"median-price range {cat_price['median_price'].max() / cat_price['median_price'].min():.0f}×"
    ),
    height=650, width=1250, template="plotly_white",
    margin=dict(l=180, t=100),
)

fig.show()
fig.write_image(PROJECT_ROOT / "reports" / "figures" / "13_s1_category_vs_price.png", scale=2)

### §1 — Result

**Test 1 — Welch's ANOVA (category → log10_price)**
- Sample: n = 96,398 across 51 categories (≥100 filter)
- Welch's F(50, 6554) = 340, p ≈ 0
- **ω² = 0.129 — moderate (edge of large; large threshold = 0.14)**

**Test 2 — Games-Howell post-hoc**
- Total pairs: 1,275
- **Significant pairs (p < 0.05): 899 (70.5%)**
- Top gaps: computers vs electronics (34×), vs telephony (27×), vs food/drink (21–23×)
- Hedges g values of 3.5–4.6 indicate near-non-overlapping distributions

**Median-price range:** R$21.90 (electronics) to R$1,200 (computers) — a 55× span.

**Top 5 categories by median price:** computers (R$1,200), agro_industry (R$345),
home_appliances_2 (R$230), construction_tools_lights (R$164), office_furniture (R$163).

**Bottom 5 categories by median price:** electronics (R$22), telephony (R$30),
food_drink (R$41), drinks (R$47), books_general_interest (R$49).

**Interpretation.** Product category is a strong predictor of price, with 71%
of category pairs significantly different after Games-Howell correction and
an ω² near the large-effect boundary. This contrasts directly with Q3: 71%
of category pairs differ on **price** but only 15% differ on **satisfaction**.
Categories are heterogeneous on what they cost but homogeneous on how
satisfied customers are once delivered.

**Cross-test pattern: office_furniture.** The only category that appears in
both the top-5 expensive list (§1) and the bottom-5 satisfaction list (Q3).
At n = 1,270 orders with median R$163 and mean review 3.65, it breaks the
otherwise clean price-satisfaction independence observed elsewhere in the
catalog. Plausible mechanisms (bulk-shipping damage, assembly complexity,
higher expectations at higher prices) cannot be confirmed from this data
but make it the first category worth investigating in a retention study.

**Methodological note.** Log10 transform chosen because raw prices are
heavy-tailed (R$1 to R$13,440); compressing the tail makes category means
interpretable and partially addresses variance heterogeneity. Welch's ANOVA
rather than standard ANOVA because variance equality does not hold across
categories (confirmed by inspection — luxury categories have both higher
means and higher variances). Games-Howell is the variance-robust post-hoc
designed to pair with Welch's.